In [ ]:
import uuid
import pandas as pd
import psycopg2
import sys
import os
from qdrant_client import QdrantClient
from qdrant_client.http import models
from langchain_text_splitters import RecursiveCharacterTextSplitter
sys.path.append(os.path.abspath(os.path.join('..')))
from final_project.connect_to_google_drive import get_sheets_client, SHEET_ID
from fastembed import SparseTextEmbedding
from dotenv import load_dotenv
from openai import OpenAI

In [ ]:
QDRANT_HOST = "localhost"
QDRANT_PORT = 6333

In [ ]:
client = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)

In [ ]:
sheets_client = get_sheets_client()
sheet = sheets_client.open_by_key(SHEET_ID).sheet1
df = pd.DataFrame(sheet.get_all_records())
to_sync = df[(df['status'] == 'Success') & (df['vector_db_sync'] == 'No')]

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

In [ ]:
def get_text_from_postgres(file_id):
    try:
        conn = psycopg2.connect(
            dbname="ucu_rag_db", user="user", password="password", host="127.0.0.1", port=5432
        )
        cur = conn.cursor()
        cur.execute(
            "SELECT markdown_content FROM processed_documents WHERE google_drive_id = %s", 
            (file_id,)
        )
        result = cur.fetchone()
        cur.close()
        conn.close()
        return result[0] if result else None
    except Exception as e:
        print(f"ERROR: {e}")
        return None

In [ ]:
def upload_to_qdrant_openai(to_sync, openai_client, collection, sparse_model=None):
    for _, row in to_sync.iterrows():
            file_id = row['google_drive_id']
            file_name = row['file_name']
            
            text = get_text_from_postgres(file_id)

            if text:
                chunks = text_splitter.split_text(text)
                for i, chunk in enumerate(chunks):
                    response = openai_client.embeddings.create(
                        input=[chunk.replace("\n", " ")],
                        model="text-embedding-3-large"
                    )
                    dense_vector = response.data[0].embedding
                    vectors_to_upload = {"default": dense_vector}
                    if sparse_model:
                        sparse_emb = list(sparse_model.embed([chunk]))[0]
                        vectors_to_upload["text_sparse"] = models.SparseVector(
                            indices=sparse_emb.indices.tolist(),
                            values=sparse_emb.values.tolist()
                        )
                    point_id = str(uuid.uuid5(uuid.NAMESPACE_DNS, f"{file_id}_{i}"))
                    client.upsert(
                        collection_name=collection,
                        points=[
                            models.PointStruct(
                                id=point_id,
                                vector=vectors_to_upload,
                                payload={
                                    "file_id": file_id,
                                    "file_name": file_name,
                                    "text": chunk,
                                    "chunk_index": i
                                }
                            )
                        ]
                    )
                print(f"File {file_name} uploaded into Qdrant")

In [ ]:
load_dotenv()

In [ ]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

openai_client = OpenAI(api_key=OPENAI_API_KEY)

In [ ]:
MODEL_SPARSE = SparseTextEmbedding(model_name="Qdrant/bm25")

In [ ]:
COLLECTION_OPENAI = "ucu_documents_openai"

if not client.collection_exists(COLLECTION_OPENAI):
    client.create_collection(
        collection_name=COLLECTION_OPENAI,
        vectors_config={
            "default": models.VectorParams(
                size=3072,
                distance=models.Distance.COSINE
            )
        },
        sparse_vectors_config={
            "text_sparse": models.SparseVectorParams(
                index=models.SparseIndexParams(
                    on_disk=False,
                )
            )
        }
    )

In [ ]:
upload_to_qdrant_openai(to_sync=to_sync, openai_client=openai_client, collection=COLLECTION_OPENAI, sparse_model=MODEL_SPARSE)